In [ ]:
# Cell 1 — 공통 실행 환경 설정
# 이 Cell은 모든 tutorial Notebook의 첫 번째 Cell이며 여러 번 실행해도 안전합니다.

from pathlib import Path
import importlib.util
import os
import subprocess
import sys

# Google Drive의 project folder가 다르면 이 값만 수정합니다.
COLAB_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/DiffDodree"
)

# 필요하면 환경변수로 project root를 직접 지정할 수 있습니다.
PROJECT_ROOT_OVERRIDE = os.environ.get("GUGAKGEN_PROJECT_ROOT")


def find_local_project_root(start: Path) -> Path:
    """현재 위치에서 pyproject.toml을 기준으로 project root를 찾습니다."""
    candidates = [start.resolve(), *start.resolve().parents]

    for candidate in candidates:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "project root를 찾지 못했습니다. "
        "GUGAKGEN_PROJECT_ROOT를 지정하거나 project folder에서 Notebook을 실행하세요."
    )


try:
    # 순수 local Python에는 상위 ``google`` namespace가 없을 수 있습니다.
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    # 이미 mount되어 있으면 기존 mount를 그대로 사용합니다.
    drive.mount("/content/drive", force_remount=False)

if PROJECT_ROOT_OVERRIDE:
    project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
elif IN_COLAB:
    project_root = COLAB_PROJECT_ROOT.resolve()
else:
    project_root = find_local_project_root(Path.cwd())

if not project_root.exists():
    raise FileNotFoundError(
        f"project folder가 존재하지 않습니다: {project_root}\n"
        "COLAB_PROJECT_ROOT 또는 GUGAKGEN_PROJECT_ROOT를 확인하세요."
    )

bootstrap_script = project_root / "scripts" / "bootstrap_notebook.py"

if not bootstrap_script.exists():
    raise FileNotFoundError(
        f"Notebook bootstrap script를 찾지 못했습니다: {bootstrap_script}"
    )

environment_profile = "colab" if IN_COLAB else "local"

# 필요한 dependency만 확인하고 누락된 package만 설치합니다.
# Colab profile에서는 Colab이 제공한 CUDA용 PyTorch를 교체하지 않습니다.
subprocess.run(
    [
        sys.executable,
        str(bootstrap_script),
        "--project-root",
        str(project_root),
        "--profile",
        environment_profile,
        "--install-missing",
    ],
    check=True,
)

# 현재 kernel에서 src package를 즉시 import할 수 있게 합니다.
src_root = project_root / "src"

if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

os.environ["GUGAKGEN_PROJECT_ROOT"] = str(project_root)
os.chdir(project_root)

from gugakgen.device import resolve_device
from gugakgen.reproducibility import set_global_seed

runtime = resolve_device(preference="auto")
device = runtime.device

SEED = 2026
set_global_seed(SEED)

print("환경 설정 완료")
print(f"- 실행 환경: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"- Project root: {project_root}")
print(f"- Device: {device}")
print(f"- AMP enabled: {runtime.amp_enabled}")
print(f"- Seed: {SEED}")

# 02. Transition Graph와 환입 relation

이 Notebook은 전문가가 구분한 마디를 A–I family sequence로 읽고, 인접 마디 사이의
local transition Graph와 떨어진 장 사이의 환입 relation을 함께 확인합니다.

local Graph는 관찰된 family transition을 나타내며, 환입 relation은 제1장 material과
제4장 환입 구간 사이의 non-local correspondence를 별도로 표현합니다.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.patches import FancyArrowPatch

from gugakgen.config import load_yaml
from gugakgen.workflows import load_piece_context

train_config = load_yaml(project_root / "configs/train_maskgit.yaml")
context = load_piece_context(
    train_config["piece_config"],
    project_root=project_root,
    model_section=train_config["model"],
)

print("마디 수:", context.tokens.num_measures)
print("family:", context.graph.node_order)
print("환입 pair:", [(source + 1, target + 1) for source, target in context.hwanip_relations.pairs])

In [ ]:
# family와 local transition을 table로 확인합니다.
family_rows = []
for name in context.graph.node_order:
    node = context.graph.metadata["nodes"][name]
    family_rows.append(
        {
            "family": name,
            "마디 수": node["count"],
            "medoid 마디": node["medoid_measure"],
            "소속 마디": node["member_measures"],
        }
    )
display(pd.DataFrame(family_rows))

edge_table = pd.DataFrame(
    [
        {
            "source": edge.source,
            "target": edge.target,
            "count": edge.count,
            "probability": edge.probability,
        }
        for edge in context.graph.edges
    ]
).sort_values(["source", "probability"], ascending=[True, False])
display(edge_table)

In [ ]:
# local transition Graph를 시각화합니다.
graph_nx = nx.DiGraph()
for name in context.graph.node_order:
    graph_nx.add_node(name)
for edge in context.graph.edges:
    graph_nx.add_edge(edge.source, edge.target, weight=edge.count, probability=edge.probability)

positions = nx.circular_layout(graph_nx)
weights = [0.8 + 0.35 * graph_nx[u][v]["weight"] for u, v in graph_nx.edges]

fig, ax = plt.subplots(figsize=(8, 7))
nx.draw_networkx_nodes(graph_nx, positions, node_size=1500, node_color="#dbe4ff", ax=ax)
nx.draw_networkx_labels(graph_nx, positions, font_size=12, font_weight="bold", ax=ax)
nx.draw_networkx_edges(
    graph_nx,
    positions,
    width=weights,
    alpha=0.55,
    arrows=True,
    arrowsize=18,
    connectionstyle="arc3,rad=0.08",
    ax=ax,
)
ax.set_title("양청도드리 family transition Graph")
ax.axis("off")
plt.show()

In [ ]:
# 장 경계와 non-local 환입 relation을 하나의 timeline에서 확인합니다.
family_ids = context.tokens.family
family_names = [context.vocabulary.decode_family(int(value)) for value in family_ids]
family_y = {name: index for index, name in enumerate(context.graph.node_order)}
x = np.arange(1, context.tokens.num_measures + 1)
y = np.asarray([family_y[name] for name in family_names])

fig, ax = plt.subplots(figsize=(15, 5))
ax.step(x, y, where="mid", color="#495057", linewidth=1.2)
ax.scatter(x, y, c=context.tokens.chapter, cmap="tab10", s=30, zorder=3)

for chapter in context.config["chapters"]:
    start, end = chapter["measures"]
    ax.axvline(start - 0.5, color="#adb5bd", linewidth=0.8)
    ax.text((start + end) / 2, len(family_y) + 0.25, f"제{chapter['chapter']}장", ha="center")

for source, target in context.hwanip_relations.pairs:
    arc = FancyArrowPatch(
        (source + 1, family_y[family_names[source]] + 0.15),
        (target + 1, family_y[family_names[target]] + 0.15),
        connectionstyle="arc3,rad=-0.35",
        arrowstyle="-|>",
        mutation_scale=12,
        color="#e03131",
        alpha=0.8,
    )
    ax.add_patch(arc)

ax.set_yticks(range(len(family_y)), context.graph.node_order)
ax.set_xlim(0.5, context.tokens.num_measures + 0.5)
ax.set_ylim(-0.7, len(family_y) + 0.8)
ax.set_xlabel("마디")
ax.set_ylabel("family")
ax.set_title("장 경계, family path, 환입 relation")
ax.grid(axis="x", alpha=0.1)
plt.show()

In [ ]:
# 원곡 path와 환입 relation의 deterministic validation 결과를 확인합니다.
transition_validity = context.graph.transition_validity(context.tokens.family)
family_return = context.hwanip_relations.family_satisfaction(context.tokens.family)
surface_signatures = [measure.signature() for measure in context.score.measures]
surface_return = context.hwanip_relations.surface_satisfaction(surface_signatures)

display(
    pd.DataFrame(
        [
            {"metric": "transition validity", "value": transition_validity},
            {"metric": "환입 family satisfaction", "value": family_return},
            {"metric": "환입 surface satisfaction", "value": surface_return},
        ]
    )
)

## 해석 주의

transition Graph와 환입 relation은 서로 다른 구조 층위입니다. transition Graph의
validity가 높다고 해서 새로운 제1장 material이 제4장에 관계적으로 돌아온다는 뜻은
아닙니다. 그 주장은 `05_hwanip_counterfactual_experiment.ipynb`의 Counterfactual
experiment로 별도 검증합니다.